In [1]:
%load_ext autoreload
%autoreload 2
from seek_code import SEEK
from backbone import Backbone
from retrieval import Retrieval
from projector import Projector
from data_handler import EleHandler
from concept_head_tunneled import ConceptHeadTunneled, categorical_CE_loss, CrossedHeadNN, MultiHeadNN

from pathlib import Path
from torch.utils.data import DataLoader, Subset

/data/vision/beery/scratch/antoine/micromamba/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
code = 'DualRetrieval'

dataset = EleHandler(subset='IDI_6')
train_indices, test_indices = dataset.split_perpendicular_to_elephants_and_encounters(split_sizes=[0.5,0.5], hour_delta = 0.2)

train_subset = Subset(dataset, train_indices)
test_subset = Subset(dataset, test_indices)

train_loader = DataLoader(train_subset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_subset, batch_size=64, shuffle=True)

## First sanity check : let's check that our projector fine-tunes MD as well as the projector

In [11]:
code = 'finetuning-MD'
r = Retrieval(experiment_code=code)

# Train with the backbone
MegaDescriptor_1 = Backbone(model_name="MegaDescriptor", pretraining="savannah_elephants", experiment_code=code, lr = 1e-5)
MegaDescriptor_1.train(train_loader,test_loader, num_epochs=20)
r.evaluate_model(MegaDescriptor_1, train_loader,test_loader)


/data/vision/beery/scratch/antoine/CBM_reid/methods/backbone.py:33: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(weight_path, map_location=self.devi

Backbone parameters require gradients: True
Epoch 1/20 - Train Loss: 7.0516, BATCH-Recall@1: 0.55%, --- EPOCH - Recall@1: 7.16%, Recall@5: 10.97%, Recall@20: 19.50%, Recall@100: 41.74%
Epoch 2/20 - Train Loss: 6.7337, BATCH-Recall@1: 0.50%, --- EPOCH - Recall@1: 6.78%, Recall@5: 10.36%, Recall@20: 20.11%, Recall@100: 42.12%
Epoch 3/20 - Train Loss: 6.5052, BATCH-Recall@1: 0.46%, --- EPOCH - Recall@1: 6.70%, Recall@5: 10.59%, Recall@20: 20.72%, Recall@100: 42.80%
Epoch 4/20 - Train Loss: 6.3964, BATCH-Recall@1: 0.76%, --- EPOCH - Recall@1: 6.85%, Recall@5: 11.27%, Recall@20: 21.17%, Recall@100: 43.95%
Epoch 5/20 - Train Loss: 6.2087, BATCH-Recall@1: 0.81%, --- EPOCH - Recall@1: 6.78%, Recall@5: 12.19%, Recall@20: 21.78%, Recall@100: 43.34%
Epoch 6/20 - Train Loss: 6.0646, BATCH-Recall@1: 1.01%, --- EPOCH - Recall@1: 6.63%, Recall@5: 12.57%, Recall@20: 21.10%, Recall@100: 44.17%
Epoch 7/20 - Train Loss: 5.9283, BATCH-Recall@1: 0.66%, --- EPOCH - Recall@1: 6.85%, Recall@5: 12.41%, Recall@

wandb: Currently logged in as: antoinesalaun (antoinesalaun-massachusetts-institute-of-technology) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Train Recalls:
Recall@1: 11.46%
Recall@5: 22.72%
Recall@10: 28.37%
Recall@20: 37.41%
Recall@100: 65.17%
Test Recalls:
Recall@1: 30.46%
Recall@5: 44.02%
Recall@10: 50.88%
Recall@20: 57.96%
Recall@100: 79.66%


wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.


/data/vision/beery/scratch/antoine/CBM_reid/methods/backbone.py:33: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(weight_path, map_location=self.devi

Model loaded from /data/vision/beery/scratch/antoine/CBM_reid/weights/concept_w.pt
loading weights for the concept head
Parameters requiring gradients - Projector: False, Backbone_for_concepts: True, Backbone: False, Concept_head: False
Embeddings norm: 47.055641174316406, Projected concepts norm: 56.456443786621094


UnboundLocalError: cannot access local variable 'epoch_val_recall' where it is not associated with a value

In [ ]:
# Train with the projector
pr = Projector(loss_type ='ArcFace', lr=1e-5, scale=64, margin=0.5, experiment_code=code, intervention_fn=None, alpha = 0, print_every=1)
MD_for_concepts = Backbone(model_name="MegaDescriptor", pretraining="backbone_for_concepts_w", experiment_code=code)
MegaDescriptor_2 = Backbone(model_name="MegaDescriptor", pretraining="savannah_elephants", experiment_code=code, lr = 1e-5)
concept_head = ConceptHeadTunneled(loss=categorical_CE_loss, experiment_code = code, layer = CrossedHeadNN(), reset_weights=False)

concept_head.freeze()
MegaDescriptor_2.unfreeze()
MD_for_concepts.freeze()
pr.freeze()

pr.train(train_loader, test_loader, MD_for_concepts, MegaDescriptor_2, concept_head, num_epochs=20)
r.evaluate_model(pr, train_loader, test_loader, ba = MegaDescriptor_2, ch = concept_head, backbone_for_concepts = MD_for_concepts)

wandb: Currently logged in as: antoinesalaun (antoinesalaun-massachusetts-institute-of-technology) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.


/data/vision/beery/scratch/antoine/CBM_reid/methods/backbone.py:33: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(weight_path, map_location=self.devi

Model loaded from /data/vision/beery/scratch/antoine/CBM_reid/weights/concept_w.pt
loading weights for the concept head
Parameters requiring gradients - Projector: False, Backbone_for_concepts: False, Backbone: True, Concept_head: False


## Proof of concept

In [ ]:
code = 'md_only-alpha=0-lr=5e-6'

MD_for_concepts = Backbone(model_name="MegaDescriptor", pretraining="backbone_for_concepts_w", experiment_code=code)
MD_finedtuned = Backbone(model_name="MegaDescriptor", pretraining="savannah-elephants", experiment_code=code)
pr = Projector(loss_type ='ArcFace', lr=5e-6, scale=64, margin=0.5, experiment_code=code, intervention_fn=SEEK.correct_or_soft, alpha = 0, network_type = 'small')
concept_head = ConceptHeadTunneled(loss=categorical_CE_loss, experiment_code = code, layer = CrossedHeadNN(), reset_weights=False)

# Freeze other models and unfreeze the projector
MD_for_concepts.freeze()
MD_finedtuned.unfreeze()
concept_head.freeze()
pr.freeze()

# Train the model
pr.train(
    train_loader, 
    test_loader, 
    backbone_for_concepts=MD_for_concepts, 
    backbone=MD_finedtuned, 
    concept_head=concept_head, 
    num_epochs=400
)

# Define intervention functions for testing
test_intervention_fns = {
    "100% Correction": SEEK.oracle_correction,
    "50% Correction + soft": SEEK.correct_or_soft,
    "0% Correction Hard": SEEK.hard,
    "0% Correction": None
}

# Run the structured test method
pr.test(
    backbone_for_concepts=MD_for_concepts, 
    backbone=MD_finedtuned, 
    concept_head=concept_head, 
    train_loader=train_loader, 
    test_loader=test_loader, 
    intervention_fns=test_intervention_fns
)

In [ ]:
code = 'alpha=0.75-lr=5e-6-correct-or-sosft'

MD_for_concepts = Backbone(model_name="MegaDescriptor", pretraining="backbone_for_concepts_w", experiment_code=code)
#MD_finedtuned = Backbone(model_name="MegaDescriptor", pretraining="MD_finetuned_w", experiment_code=code)
pr = Projector(loss_type ='ArcFace', lr=5e-6, scale=64, margin=0.5, experiment_code=code, intervention_fn=SEEK.correct_or_soft, alpha = 0.75, network_type = 'small')
concept_head = ConceptHeadTunneled(loss=categorical_CE_loss, experiment_code = code, layer = CrossedHeadNN(), reset_weights=False)

# Freeze other models and unfreeze the projector
MD_for_concepts.freeze()
MD_finedtuned.freeze()
concept_head.freeze()
pr.unfreeze()

# Train the model
pr.train(
    train_loader, 
    test_loader, 
    backbone_for_concepts=MD_for_concepts, 
    backbone=MD_finedtuned, 
    concept_head=concept_head, 
    num_epochs=400
)

# Define intervention functions for testing
test_intervention_fns = {
    "100% Correction": SEEK.oracle_correction,
    "50% Correction + soft": SEEK.correct_or_soft,
    "0% Correction Hard": SEEK.hard,
    "0% Correction": None
}

# Run the structured test method
pr.test(
    backbone_for_concepts=MD_for_concepts, 
    backbone=MD_finedtuned, 
    concept_head=concept_head, 
    train_loader=train_loader, 
    test_loader=test_loader, 
    intervention_fns=test_intervention_fns
)

## Training with soft concepts corrected 50% of the time 

In [ ]:
code = 'alpha=0.75-lr=1e-5-correct-or-sosft'

MD_for_concepts = Backbone(model_name="MegaDescriptor", pretraining="backbone_for_concepts_w", experiment_code=code)
MD_finedtuned = Backbone(model_name="MegaDescriptor", pretraining="MD_finetuned_w", experiment_code=code)
pr = Projector(loss_type ='ArcFace', lr=1e-5, scale=64, margin=0.5, experiment_code=code, intervention_fn=SEEK.correct_or_soft, alpha = 0.75)

import torch.nn as nn
pr.layer = nn.Sequential(
            nn.Linear(63, 2304),
            nn.LeakyReLU()
        ).to('cuda')
import torch
with torch.no_grad():
    for i, param in enumerate(pr.layer.parameters()):
        if len(param.shape) == 2:  # Only initialize weight matrices
            param.copy_(torch.eye(param.shape[0], param.shape[1]).to(param.device))

concept_head = ConceptHeadTunneled(loss=categorical_CE_loss, experiment_code = code, layer = CrossedHeadNN(), reset_weights=False)

#r = Retrieval(experiment_code=code)
#r.evaluate_model(concept_head, ba= MD_for_concepts, train_loader=train_loader, test_loader=test_loader, show_matches=False, show_plot=False, show_tsne=False, intervention_fn = SEEK.oracle_correction)

# Freeze other models and unfreeze the projector
MD_for_concepts.freeze()
MD_finedtuned.freeze()
concept_head.freeze()
pr.unfreeze()

# Train the model
pr.train(
    train_loader, 
    test_loader, 
    backbone_for_concepts=MD_for_concepts, 
    backbone=MD_finedtuned, 
    concept_head=concept_head, 
    num_epochs=30
)

# Define intervention functions for testing
test_intervention_fns = {
    "100% Correction": SEEK.oracle_correction,
    "50% Correction + soft": SEEK.correct_or_soft,
    "0% Correction": None,
    "0% Correction Hard": SEEK.hard
}

# Run the structured test method
pr.test(
    backbone_for_concepts=MD_for_concepts, 
    backbone=MD_finedtuned, 
    concept_head=concept_head, 
    train_loader=train_loader, 
    test_loader=test_loader, 
    intervention_fns=test_intervention_fns
)

/data/vision/beery/scratch/antoine/CBM_reid/methods/backbone.py:33: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(weight_path, map_location=self.devi

/data/vision/beery/scratch/antoine/CBM_reid/methods/concept_head_tunneled.py:388: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.load_state_dict(torch.load(path, map_loc

Model loaded from /data/vision/beery/scratch/antoine/CBM_reid/weights/concept_w.pt
loading weights for the concept head
Parameters requiring gradients - Projector: True, Backbone_for_concepts: False, Backbone: False, Concept_head: False
Embeddings norm: 51.14790344238281, Projected concepts norm: 14.8927583694458
Epoch 1/30 | Norms: bakcbone 3739.54048538208 - proj 11.41034197807312 | Loss: 7.1270 | Batch R@1: 4.91% | Train: R@1=48.6% R@5=67.4% R@20=82.3% R@100=95.1% | Val: R@1=28.1% R@5=41.1% R@10=47.4% R@20=53.8% R@100=72.3%
Embeddings norm: 52.07807159423828, Projected concepts norm: 14.374872207641602
Epoch 2/30 | Norms: bakcbone 3739.54048538208 - proj 11.410743474960327 | Loss: 7.0303 | Batch R@1: 4.44% | Train: R@1=48.5% R@5=67.0% R@20=82.3% R@100=95.1% | Val: R@1=28.4% R@5=39.8% R@10=46.4% R@20=53.2% R@100=74.6%
Embeddings norm: 53.403785705566406, Projected concepts norm: 14.724599838256836
Epoch 3/30 | Norms: bakcbone 3739.54048538208 - proj 11.411388874053955 | Loss: 6.9707 

BATCH-Recall@1,▆▄▇▆▆█▆▇▄█▃▇▆▆█▃▅▆▃▅▇▇▆▃▆▇▁▅▃▄
backbone_norm,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
projector_norm,▁▁▁▁▁▂▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▆▆▆▆▇▇▇██
train-Recall@1,█████▇▇▆▆▆▅▅▅▄▄▄▃▃▂▂▂▂▂▁▁▁▁▁▁▁
train-Recall@100,█████▇▇▇▇▆▆▆▆▅▅▅▅▄▄▃▃▃▂▂▂▂▂▂▁▁
train-Recall@20,█████▇▇▇▆▆▅▅▄▄▄▃▃▃▃▂▂▂▂▂▂▁▁▁▁▁
train-Recall@5,█▇▇▇▇▇▆▆▆▆▅▅▅▄▄▃▃▃▂▂▂▂▁▁▁▁▁▁▁▁
train_loss,█▇▇▇▆▆▅▅▅▅▄▄▄▄▄▃▃▃▃▃▂▂▂▂▂▂▂▁▁▁
val-Recall@1,▃▃█▄▅▃▆▇▅▂▄▇▂▆▃▅▅▇▁▃▇▃▄▁▅▂▂▂▅▄
val-Recall@10,▄▃█▄▄▄▆▇▄▂▅▆▂▆▃▆▄▆▁▃▆▃▅▁▃▂▃▃▅▅


{'100% Correction': {1: 0.4135567402894136,
  5: 0.5833968012185834,
  10: 0.6488956587966489,
  20: 0.7151561309977151,
  100: 0.8789032749428789},
 '50% Correction + soft': {1: 0.32444782939832445,
  5: 0.4607768469154608,
  10: 0.5163747143945163,
  20: 0.5917745620715917,
  100: 0.7829398324447829},
 '0% Correction': {1: 0.24980959634424982,
  5: 0.3488194973343488,
  10: 0.4013709063214014,
  20: 0.4798172124904798,
  100: 0.6877380045696877},
 '0% Correction Hard': {1: 0.376999238385377,
  5: 0.5323686214775324,
  10: 0.5978674790555979,
  20: 0.6694592536176694,
  100: 0.8347296268088348}}

## Training with hard concepts corrected 50% of the time 

In [ ]:
code = 'correct_or_hard_@50%'

MD_for_concepts = Backbone(model_name="MegaDescriptor", pretraining="backbone_for_concepts_w", experiment_code=code)
MD_finedtuned = Backbone(model_name="MegaDescriptor", pretraining="MD_finetuned_w", experiment_code=code)
pr = Projector(lr=1e-3, scale=64, margin=0.5, experiment_code=code, intervention_fn=SEEK.oracle_correction)
concept_head = ConceptHeadTunneled(loss=categorical_CE_loss, experiment_code = code, layer = CrossedHeadNN(), reset_weights=False)

# Freeze other models and unfreeze the projector
MD_for_concepts.freeze()
MD_finedtuned.freeze()
concept_head.freeze()
pr.unfreeze()

# Train the model
pr.train(
    train_loader, 
    test_loader, 
    backbone_for_concepts=MD_for_concepts, 
    backbone=MD_finedtuned, 
    concept_head=concept_head, 
    num_epochs=100
)

# Define intervention functions for testing
test_intervention_fns = {
    "100% Correction": SEEK.oracle_correction,
    "50% Correction + soft": correct_or_soft,
    "50% Correction + hard": correct_or_hard,
    "0% Correction": None,
    "0% Correction Hard": hard
}

# Run the structured test method
pr.test(
    backbone_for_concepts=MD_for_concepts, 
    backbone=MD_finedtuned, 
    concept_head=concept_head, 
    train_loader=train_loader, 
    test_loader=test_loader, 
    intervention_fns=test_intervention_fns
)

## Training with 0% corrected soft concepts

In [ ]:

# Initialize Projector
pr = Projector(lr=5e-4, scale=64, margin=0.5, experiment_code='soft_corrected', intervention_fn=None)
concept_head = ConceptHeadTunneled(loss=categorical_CE_loss, experiment_code = code, layer = CrossedHeadNN(), reset_weights=False)

# Freeze other models and unfreeze the projector
MD_for_concepts.freeze()
concept_head.freeze()
pr.unfreeze()

# Train the model
pr.train(
    train_loader, 
    test_loader, 
    backbone_for_concepts=MD_for_concepts, 
    backbone=MD_finedtuned, 
    concept_head=concept_head, 
    num_epochs=100
)

# Define intervention functions for testing
test_intervention_fns = {
    "100% Correction": SEEK.oracle_correction,
    "50% Correction + soft": correct_or_soft,
    "50% Correction + hard": correct_or_hard,
    "0% Correction": None,
    "0% Correction Hard": hard
}

# Run the structured test method
pr.test(
    backbone_for_concepts=MD_for_concepts, 
    backbone=MD_finedtuned, 
    concept_head=concept_head, 
    train_loader=train_loader, 
    test_loader=test_loader, 
    intervention_fns=test_intervention_fns
)

## A tester :
- Soft concepts or correct 1/2
- Aggregated on Subject SEEK
- Just ELE_SEEK
- Corrected ELE_SEEK, test with diffferent level of 